In [ ]:
# CYR-GPU-011 — CELL 0: FREEZE / VERIFY / CALIBRATE / RESOLVE
import json, os, subprocess, sys
from pathlib import Path
REPO = Path('/content/An-Ra-the-new-AGI')
BRANCH = 'cymek-500m-readiness'
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
PREREG_PATH = REPO / 'docs/cymek/experiments/CYR-GPU-011/PREREGISTRATION.json'
if not PREREG_PATH.exists():
    raise RuntimeError('CYR-GPU-011 is not preregistered yet; do not execute this notebook')
PREREG = json.loads(PREREG_PATH.read_text('utf-8'))
EXTERNAL = Path('/content/CYR-GPU-011-PREREGISTRATION.json')
EXTERNAL.write_text(json.dumps(PREREG, indent=2, sort_keys=True), encoding='utf-8')
EXECUTABLE_SHA = PREREG['executable_sha']
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXECUTABLE_SHA], check=True)
HEAD = subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert HEAD == EXECUTABLE_SHA, (HEAD, EXECUTABLE_SHA)
for relative, expected_blob in PREREG['executable_blobs'].items():
    actual_blob = subprocess.run(['git','-C',str(REPO),'hash-object',relative], check=True, capture_output=True, text=True).stdout.strip()
    assert actual_blob == expected_blob, f'blob mismatch: {relative}'
print('frozen executable verified:', HEAD)
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers','pytest','numpy'], check=True)
sys.path.insert(0, str(REPO))
subprocess.run([sys.executable,'-m','py_compile','v5_experiments/cyr_gpu011.py','anra_v5/cyr_gpu011_run.py','anra_v5/cyr_gpu011_entry.py','anra_v5/cyr_gpu011_optimizer_compat.py'], check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_v5_cyr_gpu011.py','tests/test_v5_cyr_gpu011_entry.py','-q'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CYR-GPU-011 requires a Google Colab CUDA GPU')
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from anra_v5.cyr_gpu011_entry import production_tokenizer, calibrate_all
from v5_experiments import cyr_gpu011 as core
prod_tok, prod_identity = production_tokenizer(REPO)
assert prod_identity['artifact_sha256'] == PREREG['tokenizer']['artifact_sha256']
manifest_path = REPO / 'docs/cymek/experiments/CYR-GPU-011/ARK002B_TASK_MANIFEST.json'
data = core.load_ark002b_manifest(manifest_path)
assert data['source_split_sha256'] == PREREG['data']['split_sha256']
assert data['source_blob_sha'] == PREREG['data']['manifest_blob_sha']
compact_tok = core.CompactCharTokenizer()
compact_spec = core.research_small_spec(compact_tok.vocabulary_size)
production_spec = core.research_small_spec(prod_identity['vocabulary_size'])
receipts = core.model_receipts()
assert receipts['COMPACT_BRIDGE']['parameters'] == compact_spec.parameter_receipt().total
assert receipts['PRODUCTION_BRIDGE']['parameters'] == production_spec.parameter_receipt().total
prod_special = {'pad_id':prod_identity['pad_id'],'bos_id':prod_identity['bos_id'],'eos_id':prod_identity['eos_id']}
CALIBRATIONS = calibrate_all(production_tok=prod_tok, production_special=prod_special, compact_tok=compact_tok, compact_special=compact_tok.special, production_spec=production_spec, compact_spec=compact_spec, train_rows=data['train'], eval_rows=data['dev_controller'], torch=torch, device=DEVICE)
Path('/content/CYR-GPU-011-CALIBRATIONS.json').write_text(json.dumps(CALIBRATIONS, indent=2, sort_keys=True))
print('CALIBRATION SUMMARY')
for name, rec in sorted(CALIBRATIONS.items()):
    print(name, rec.get('status'), 'batch=',rec.get('batch_rows'), 'updates/s=',round(float(rec.get('training_updates_per_sec',0)),3), 'semantic_rows/s=',round(float(rec.get('semantic_rows_per_sec',0)),1), 'gen_ex/s=',round(float(rec.get('generation_examples_per_sec',0)),1), 'peak_GB=',round(float(rec.get('peak_vram_gb',0)),2))
RESOLVED = core.resolve_from_calibrations(CALIBRATIONS)
core.validate_resolved(RESOLVED)
Path('/content/CYR-GPU-011-RESOLVED.json').write_text(json.dumps(RESOLVED, indent=2, sort_keys=True))
print(json.dumps(RESOLVED, indent=2))
print('Important: production nulls are only called representation divergence if actual semantic exposure reaches the compact G90 exposure.')
print('CYR-GPU-011 PREEXECUTION GATE: PASS')


In [ ]:
# CYR-GPU-011 — CELL 1: LONG ONE-SHOT RUN / DRIVE DURABILITY
import json, sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/An-Ra-the-new-AGI')
sys.path.insert(0, str(REPO))
DEVICE = torch.device('cuda')
assert torch.cuda.is_available() and DEVICE.type == 'cuda'
PREREG = json.loads(Path('/content/CYR-GPU-011-PREREGISTRATION.json').read_text())
CALIBRATIONS = json.loads(Path('/content/CYR-GPU-011-CALIBRATIONS.json').read_text())
RESOLVED = json.loads(Path('/content/CYR-GPU-011-RESOLVED.json').read_text())
OUT = Path('/content/drive/MyDrive/CYMEK/CYR-GPU-011')
OUT.mkdir(parents=True, exist_ok=True)
from anra_v5.cyr_gpu011_entry import run_campaign
try:
    campaign = run_campaign(repo=REPO, out=OUT, preregistration=PREREG, resolved=RESOLVED, calibrations=CALIBRATIONS, torch=torch, device=DEVICE, progress=lambda msg: print(msg, flush=True))
    print('DONE:', campaign['status'])
    print('VERDICT:', campaign.get('decision',{}).get('verdict'))
    print('COMPACT G90:', campaign.get('compact_bridge',{}).get('g90_confirm_update'))
    print('PRODUCTION G90:', campaign.get('production_primary',{}).get('g90_confirm_update'))
    print('PRODUCTION EXPOSURE:', round(float(campaign.get('production_primary',{}).get('ark_exposure_fraction',0)),3))
    print('BUNDLE:', campaign['bundle']['path'])
except Exception as exc:
    print('RUN FAILED, but partial evidence was packaged on Drive:', repr(exc))
    print('OUT:', OUT)
    raise


In [ ]:
# CYR-GPU-011 — CELL 2: VERIFY / DOWNLOAD
import hashlib, json
from pathlib import Path
from google.colab import files
OUT = Path('/content/drive/MyDrive/CYMEK/CYR-GPU-011')
receipt = json.loads((OUT/'campaign_receipt.json').read_text('utf-8'))
bundle = Path(receipt['bundle']['path'])
assert bundle.exists(), bundle
actual = hashlib.sha256(bundle.read_bytes()).hexdigest()
assert actual == receipt['bundle']['sha256'], 'bundle hash mismatch'
print('verified', bundle.name, actual)
print('status:', receipt['status'])
print('verdict:', receipt.get('decision',{}).get('verdict'))
print('compact rows:', receipt.get('compact_bridge',{}).get('row_presentations'), 'compact G90:', receipt.get('compact_bridge',{}).get('g90_confirm_update'))
print('production rows:', receipt.get('production_primary',{}).get('row_presentations'), 'production G90:', receipt.get('production_primary',{}).get('g90_confirm_update'))
print('production exposure fraction:', receipt.get('production_primary',{}).get('ark_exposure_fraction'))
files.download(str(bundle))
